In [0]:
%run "../00_common/01_environment_config"

In [0]:
# Databricks notebook source

# MAGIC %md
# MAGIC # Build Session Results Fact
# MAGIC
# MAGIC 1. Read Silver `results`
# MAGIC 2. Read Silver `sprints`
# MAGIC 3. Add `session_type`
# MAGIC 4. UNION Results and Sprints
# MAGIC 5. Create `is_win`, `is_podium` and `has_points`
# MAGIC 6. Write to Gold `fact_session_results`


# COMMAND ----------

# MAGIC %run "../00_common/01_environment_config"


# COMMAND ----------

from pyspark.sql import functions as F


# COMMAND ----------

# Define paths

results_path = f"{s3_root_path}/Silver/Kalashri/results"
sprints_path = f"{s3_root_path}/Silver/Kalashri/sprints"

target_path = f"{s3_root_path}/Gold/Kalashri/fact_session_results"


# COMMAND ----------


# STEP 1 - Read Silver Results and Sprints

results_df = (
    spark.read
        .format("delta")
        .load(results_path)
)

sprints_df = (
    spark.read
        .format("delta")
        .load(sprints_path)
)

display(results_df.limit(20))
display(sprints_df.limit(20))

# COMMAND ----------

# STEP 2 - Prepare Race Results

race_results_df = (
    results_df
        .select(
            "season",
            "round",
            "constructor_id",
            "driver_id",
            "grid_position",
            "completed_laps",
            "car_number",
            "points",
            "final_position",
            "final_position_text",
            "status"
        )
        .withColumn("session_type", F.lit("RACE"))
)

print("Race columns:", len(race_results_df.columns))
print(race_results_df.columns)


# COMMAND ----------

# Prepare Sprint Results

sprint_results_df = (
    sprints_df
        .select(
            "season",
            "round",
            "constructor_id",
            "driver_id",
            "grid_position",
            "completed_laps",
            "car_number",
            "points",
            "final_position",
            "final_position_text",
            "status"
        )
        .withColumn("session_type", F.lit("SPRINT"))
)

print("Sprint columns:", len(sprint_results_df.columns))
print(sprint_results_df.columns)


# COMMAND ----------

# STEP 3 - UNION Results and Sprints

results_sprints_df = (
    race_results_df
        .unionByName(sprint_results_df)
)

display(results_sprints_df.limit(20))
# COMMAND ----------

# STEP 4 - Add Derived Columns

fact_session_results_df = (
    results_sprints_df
        .withColumn(
            "is_win",
            F.col("final_position") == 1
        )
        .withColumn(
            "is_podium",
            F.col("final_position").between(1, 3)
        )
        .withColumn(
            "has_points",
            F.col("points") > 0
        )
)

display(fact_session_results_df.limit(20))


# COMMAND ----------

# STEP 5 - Write to Gold

(
    fact_session_results_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(target_path)
)


# COMMAND ----------

# STEP 6 - Validate Gold Fact

gold_fact_df = (
    spark.read
        .format("delta")
        .load(target_path)
)

display(gold_fact_df.limit(20))


# COMMAND ----------

print("Race Records   :", race_results_df.count())
print("Sprint Records :", sprint_results_df.count())
print("Gold Records   :", gold_fact_df.count())

In [0]:
results_check = (
    spark.read
        .format("delta")
        .load(f"{s3_root_path}/Silver/Kalashri/results")
)

sprints_check = (
    spark.read
        .format("delta")
        .load(f"{s3_root_path}/Silver/Kalashri/sprints")
)

print("Silver Results :", results_check.count())
print("Silver Sprints :", sprints_check.count())

In [0]:
print("Gold Race Records   :", race_results_df.count())
print("Gold Sprint Records :", sprint_results_df.count())
print("Gold Total Records  :", gold_fact_df.count())